# Chapter 2 Student Colab Notebook: Matrices

**Course:** MATH 3260 - Linear Algebra  
**Sections:** 2.1-2.6  
**Audience:** Student version

This notebook is the complete interactive teaching companion for Chapter 2. It combines theory, original examples, detailed matrix calculations, applications, visualizations, computational verification, section worksheets, and complete solutions.

> **How to use this student notebook**
>
> Read the complete theory and worked examples, run the demonstration code in order, and use the section worksheets for practice. Complete worksheet solutions are distributed separately. The notebook intentionally retains rich applications and visualizations rather than reducing the chapter to a short outline.

## Chapter map

1. **Section 2.1:** Operations with Matrices  
2. **Section 2.2:** Properties of Matrix Operations  
3. **Section 2.3:** The Inverse of a Matrix  
4. **Section 2.4:** Elementary Matrices and LU-Factorization  
5. **Section 2.5:** Markov Chains  
6. **Section 2.6:** More Applications of Matrix Operations  

Each section ends with a terminology table, a worksheet, and complete instructor solutions.

In [ ]:
# Core imports used throughout Chapter 2
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown

sp.init_printing()


def show_matrix(M, label=None):
    """Display a SymPy matrix with an optional LaTeX label."""
    M = sp.Matrix(M)
    if label:
        display(Math(rf"{label}={sp.latex(M)}"))
    else:
        display(M)
    return M


def show_augmented(M, split, label=None):
    """Display an augmented matrix with a vertical bar after `split` columns."""
    M = sp.Matrix(M)
    if split <= 0 or split >= M.cols:
        raise ValueError("split must be between 1 and the total number of columns minus 1")

    column_format = "c" * split + "|" + "c" * (M.cols - split)
    row_strings = []
    for i in range(M.rows):
        entries = [sp.latex(M[i, j]) for j in range(M.cols)]
        row_strings.append(" & ".join(entries))

    body = r" \\ ".join(row_strings)
    matrix_latex = (
        rf"\left[\begin{{array}}{{{column_format}}}"
        rf"{body}"
        rf"\end{{array}}\right]"
    )
    if label:
        display(Math(rf"{label}={matrix_latex}"))
    else:
        display(Math(matrix_latex))
    return M


def matrix_entry_product(A, B, i, j):
    """Return the row-column products contributing to entry (i,j)."""
    A, B = sp.Matrix(A), sp.Matrix(B)
    terms=[A[i,k]*B[k,j] for k in range(A.cols)]
    return terms, sp.simplify(sum(terms))

print('Chapter 2 setup complete.')

> **Example policy**
>
> All worked examples and numerical values in this notebook are original. Personal-interest themes are used selectively: Spurs, Backstreet Boys, hiking, badminton, driving, travel, and R&B music. Any prices, transition probabilities, or datasets are fictional teaching models unless explicitly stated otherwise.

# Section 2.1 - Operations with Matrices

## Learning objectives

By the end of this section, students should be able to:

- identify matrix size and individual entries;
- determine whether two matrices are equal;
- add, subtract, and scale matrices;
- determine whether a matrix product is defined and find its size;
- compute matrix products by row-column multiplication;
- write a system as $A\mathbf{x}=\mathbf{b}$;
- interpret $A\mathbf{x}$ as a linear combination of the columns of $A$;
- understand simple block or partitioned matrices.

### Definition: matrix and size

> An $m	imes n$ matrix is a rectangular array with $m$ rows and $n$ columns. The entry in row $i$, column $j$ is denoted by $a_{ij}$. Two matrices are equal only when they have the same size and every pair of corresponding entries is equal.

### Original example: equality of matrices

Let

$$
A=\begin{bmatrix}9&20\\21&1\end{bmatrix},
\qquad
B=\begin{bmatrix}9&p\\21&1\end{bmatrix}.
$$

The matrices have the same size. They are equal exactly when corresponding entries agree, so

$$
p=20.
$$

A $2\times 1$ matrix is a **column vector**, and a $1\times 2$ matrix is a **row vector**.

In [ ]:
A21 = sp.Matrix([[9,20],[21,1]])
p = sp.symbols('p')
B21 = sp.Matrix([[9,p],[21,1]])
show_matrix(A21, 'A')
show_matrix(B21, 'B')
print('Equality requires p =', sp.solve(sp.Eq(B21[0,1], A21[0,1]), p)[0])

## Addition, subtraction, and scalar multiplication

Matrices can be added or subtracted only when their sizes agree. Operations are performed entry by entry:

$$
A+B=[a_{ij}+b_{ij}],
\qquad
cA=[ca_{ij}].
$$

### Original application: fictional Spurs fan-shop inventory

Suppose two kiosks record quantities of caps, shirts, and posters during two time blocks:

$$
K_1=\begin{bmatrix}9&20&21\\12&18&15\end{bmatrix},
\qquad
K_2=\begin{bmatrix}11&16&14\\8&21&20\end{bmatrix}.
$$

Rows represent kiosks and columns represent the three products. Then

$$
K_1+K_2
=\begin{bmatrix}20&36&35\\20&39&35\end{bmatrix}.
$$

If the second time block is weighted twice in a planning score, then

$$
K_1+2K_2
=\begin{bmatrix}31&52&49\\28&60&55\end{bmatrix}.
$$

In [ ]:
K1 = sp.Matrix([[9,20,21],[12,18,15]])
K2 = sp.Matrix([[11,16,14],[8,21,20]])
show_matrix(K1, 'K_1')
show_matrix(K2, 'K_2')
show_matrix(K1+K2, 'K_1+K_2')
show_matrix(K1+2*K2, 'K_1+2K_2')

> **Common mistake**
>
> Matrix addition is not based on row-column products. Addition is entry-by-entry. Matrix multiplication, introduced next, uses row-column dot products.

## Matrix multiplication

If $A$ is $m\times n$ and $B$ is $n\times p$, then $AB$ is defined and has size $m\times p$. The inside dimensions must match:

$$
(m\times \color{blue}{n})(\color{blue}{n}\times p)=m\times p.
$$

The $(i,j)$ entry is the dot product of row $i$ of $A$ and column $j$ of $B$:

$$
(AB)_{ij}=\sum_{k=1}^{n}a_{ik}b_{kj}.
$$

### Original application: badminton training plans

Two players complete three drill types. Let

$$
D=\begin{bmatrix}1&2&0\\0&1&1\end{bmatrix}
$$

record drill counts. Let

$$
S=\begin{bmatrix}1&0\\2&1\\1&3\end{bmatrix}
$$

record two scores per drill. Then $D$ is $2\times3$, $S$ is $3\times2$, and $DS$ is $2\times2$.

For the upper-left entry,

$$
(DS)_{11}=1(1)+2(2)+0(1)=5.
$$

For the upper-right entry,

$$
(DS)_{12}=1(0)+2(1)+0(3)=2.
$$

Continuing gives

$$
DS=\begin{bmatrix}5&2\\3&4\end{bmatrix}.
$$

### Reading the multiplication visual

The $(i,j)$ entry of $AB$ is obtained by taking **row $i$ of $A$** dotted with **column $j$ of $B$**. Therefore a “row 1, column 1” annotation is identifying the two pieces used to form $(AB)_{11}$; the arrow is only a visual guide.


In [ ]:
D = sp.Matrix([[1,2,0],[0,1,1]])
S = sp.Matrix([[1,0],[2,1],[1,3]])
show_matrix(D, 'D')
show_matrix(S, 'S')
for i in range(D.rows):
    for j in range(S.cols):
        terms, total = matrix_entry_product(D,S,i,j)
        print(f'Entry ({i+1},{j+1}): {terms} -> {total}')
show_matrix(D*S, 'DS')

In [ ]:
# Visualize row-column multiplication for the same example.
fig, ax = plt.subplots(figsize=(11, 4))
ax.axis('off')

ax.text(
    0.05, 0.78,
    'D = [[1, 2, 0],\n     [0, 1, 1]]',
    fontsize=14,
    family='monospace',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray')
)

ax.text(
    0.40, 0.78,
    'S = [[1, 0],\n     [2, 1],\n     [1, 3]]',
    fontsize=14,
    family='monospace',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray')
)

ax.text(
    0.75, 0.78,
    'DS = [[5, 2],\n      [3, 4]]',
    fontsize=14,
    family='monospace',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray')
)

ax.annotate(
    'row 1 · column 1',
    xy=(0.79, 0.82),
    xytext=(0.24, 0.32),
    arrowprops=dict(arrowstyle='->', lw=1.5),
    fontsize=12
)

ax.text(0.24, 0.18, r'$1(1)+2(2)+0(1)=5$', fontsize=16)
plt.show()


## Matrix equations and column combinations

Consider

$$
A=\begin{bmatrix}
1&1&1\\
2&1&0\\
0&1&2
\end{bmatrix},
\qquad
\mathbf{x}=\begin{bmatrix}9\\20\\21\end{bmatrix}.
$$

Then

$$
A\mathbf{x}
=9\begin{bmatrix}1\\2\\0\end{bmatrix}
+20\begin{bmatrix}1\\1\\1\end{bmatrix}
+21\begin{bmatrix}1\\0\\2\end{bmatrix}
=\begin{bmatrix}50\\38\\62\end{bmatrix}.
$$

This illustrates a central interpretation:

> $A\mathbf{x}$ is a linear combination of the columns of $A$, using the entries of $\mathbf{x}$ as coefficients.

In [ ]:
A_sys = sp.Matrix([[1,1,1],[2,1,0],[0,1,2]])
x_sys = sp.Matrix([9,20,21])
b_sys = A_sys*x_sys
show_matrix(A_sys, 'A')
show_matrix(x_sys, r'\mathbf{x}')
show_matrix(b_sys, r'A\mathbf{x}')
combination = sum((x_sys[i]*A_sys[:,i] for i in range(A_sys.cols)), sp.zeros(A_sys.rows,1))
show_matrix(combination, r'9\mathbf{a}_1+20\mathbf{a}_2+21\mathbf{a}_3')

## Partitioned matrices

A matrix can be divided into blocks when the partitions are compatible. For example,

$$
A=\left[\begin{array}{cc|c}
1&1&1\\2&1&0\\0&1&2
\end{array}\right]
=\begin{bmatrix}A_1&A_2\end{bmatrix},
$$

where $A_1$ contains the first two columns and $A_2$ contains the third. Block notation is useful when a matrix has a natural structure or when several right-hand sides are solved at once.

> **Closing note**
>
> Matrix operations are not isolated rules. Addition combines compatible datasets, multiplication composes relationships, and the matrix equation connects matrix algebra to systems of linear equations and column combinations.

## Section 2.1 terminology

| Term | Meaning |
|---|---|
| Matrix | Rectangular array of numbers or expressions |
| Size / order | Number of rows by number of columns |
| Entry $a_{ij}$ | Entry in row $i$, column $j$ |
| Row vector | Matrix with one row |
| Column vector | Matrix with one column |
| Matrix equality | Same size and equal corresponding entries |
| Scalar multiple | Matrix obtained by multiplying every entry by a scalar |
| Matrix product | Row-column multiplication when inner dimensions agree |
| Matrix equation | Equation such as $A\mathbf{x}=\mathbf{b}$ |
| Partitioned matrix | Matrix divided into compatible blocks |
| Linear combination | Weighted sum of vectors or matrix columns |

## Section 2.1 worksheet

1. Determine the size of each matrix and identify the requested entry.

$$
A=\begin{bmatrix}9&20&21\\1&5&7\end{bmatrix},
\qquad B=\begin{bmatrix}4\\8\\12\end{bmatrix}.
$$

Find the sizes of $A$ and $B$, $a_{23}$, and $b_{31}$.

2. Find $p$ and $q$ so that

$$
\begin{bmatrix}p&9\\20&q\end{bmatrix}
=\begin{bmatrix}1&9\\20&21\end{bmatrix}.
$$

3. Given

$$
C=\begin{bmatrix}2&-1\\4&3\end{bmatrix},
\qquad D=\begin{bmatrix}5&2\\-1&6\end{bmatrix},
$$

find $C+D$, $C-D$, and $3C-2D$.

4. Determine whether each product is defined and state its size:

- $(3\times4)(4\times2)$
- $(2\times3)(2\times2)$
- $(1\times5)(5\times1)$

5. Compute

$$
\begin{bmatrix}1&2&1\\0&1&3\end{bmatrix}
\begin{bmatrix}2&1\\1&0\\3&2\end{bmatrix}.
$$

6. Write the system as $A\mathbf{x}=\mathbf{b}$ and express $\mathbf{b}$ as a linear combination of the columns of $A$:

$$
\begin{cases}
x+2y-z=9,\\
2x+y+3z=20.
\end{cases}
$$

# Section 2.2 - Properties of Matrix Operations

## Learning objectives

Students should be able to:

- use algebraic properties of addition and scalar multiplication;
- work with zero and identity matrices;
- use associativity and distributivity of matrix multiplication;
- recognize that matrix multiplication is generally not commutative;
- solve simple matrix equations;
- compute transposes and apply transpose identities.

## Algebra of matrix addition and scalar multiplication

For matrices of the same size and scalars $c,d$:

$$
A+B=B+A,
\qquad
A+(B+C)=(A+B)+C,
$$

$$
A+O=A,
\qquad
A+(-A)=O,
$$

$$
(c+d)A=cA+dA,
\qquad
c(A+B)=cA+cB,
$$

$$
(cd)A=c(dA),
\qquad
1A=A.
$$

### Original example: solving a matrix equation

Solve

$$
3X+A=B,
$$

where

$$
A=\begin{bmatrix}1&9\\20&21\end{bmatrix},
\qquad
B=\begin{bmatrix}10&30\\41&42\end{bmatrix}.
$$

Subtract $A$ from both sides:

$$
3X=B-A
=\begin{bmatrix}9&21\\21&21\end{bmatrix}.
$$

Multiply by $\frac13$:

$$
X=\frac13(B-A)
=\begin{bmatrix}3&7\\7&7\end{bmatrix}.
$$

In [ ]:
A22 = sp.Matrix([[1,9],[20,21]])
B22 = sp.Matrix([[10,30],[41,42]])
X22 = sp.Rational(1,3)*(B22-A22)
show_matrix(A22,'A')
show_matrix(B22,'B')
show_matrix(B22-A22,'B-A')
show_matrix(X22,'X')
show_matrix(3*X22+A22,'3X+A')

## Properties of matrix multiplication

When sizes are compatible:

$$
A(BC)=(AB)C,
$$

$$
A(B+C)=AB+AC,
\qquad
(A+B)C=AC+BC,
$$

$$
c(AB)=(cA)B=A(cB).
$$

Matrix multiplication is associative, but it is generally **not commutative**.

### Detailed associativity example

Let

$$
A=\begin{bmatrix}1&2&0\\0&1&1\end{bmatrix},
\quad
B=\begin{bmatrix}1&0\\2&1\\1&3\end{bmatrix},
\quad
C=\begin{bmatrix}2&1\\0&1\end{bmatrix}.
$$

First group as $(AB)C$:

$$
AB=\begin{bmatrix}5&2\\3&4\end{bmatrix},
$$

so

$$
(AB)C
=\begin{bmatrix}5&2\\3&4\end{bmatrix}
\begin{bmatrix}2&1\\0&1\end{bmatrix}
=\begin{bmatrix}10&7\\6&7\end{bmatrix}.
$$

Now group as $A(BC)$:

$$
BC=\begin{bmatrix}2&1\\4&3\\2&4\end{bmatrix},
$$

so

$$
A(BC)
=\begin{bmatrix}1&2&0\\0&1&1\end{bmatrix}
\begin{bmatrix}2&1\\4&3\\2&4\end{bmatrix}
=\begin{bmatrix}10&7\\6&7\end{bmatrix}.
$$

Both groupings give the same result.

In [ ]:
A_assoc=sp.Matrix([[1,2,0],[0,1,1]])
B_assoc=sp.Matrix([[1,0],[2,1],[1,3]])
C_assoc=sp.Matrix([[2,1],[0,1]])
show_matrix(A_assoc*B_assoc,'AB')
show_matrix((A_assoc*B_assoc)*C_assoc,'(AB)C')
show_matrix(B_assoc*C_assoc,'BC')
show_matrix(A_assoc*(B_assoc*C_assoc),'A(BC)')
print('Associativity verified:', (A_assoc*B_assoc)*C_assoc == A_assoc*(B_assoc*C_assoc))

### Matrix multiplication is not commutative

Let

$$
P=\begin{bmatrix}1&1\\0&1\end{bmatrix},
\qquad
Q=\begin{bmatrix}1&0\\1&1\end{bmatrix}.
$$

Then

$$
PQ=\begin{bmatrix}2&1\\1&1\end{bmatrix},
\qquad
QP=\begin{bmatrix}1&1\\1&2\end{bmatrix}.
$$

Therefore, $PQ\ne QP$. The order of transformations matters.

In [ ]:
P=sp.Matrix([[1,1],[0,1]])
Q=sp.Matrix([[1,0],[1,1]])
show_matrix(P*Q,'PQ')
show_matrix(Q*P,'QP')
print('Do they commute?', P*Q == Q*P)

## Identity and zero matrices

The identity matrix $I_n$ satisfies

$$
AI_n=A,
\qquad
I_mA=A
$$

for an $m\times n$ matrix $A$. The zero matrix satisfies $A+O=A$. However, matrix algebra differs from ordinary number algebra: $AB=O$ does not necessarily imply $A=O$ or $B=O$.

### Zero and identity matrices

The **zero matrix** $O$ has $0$ in every entry. The **identity matrix** $I_n$ has $1$s on the main diagonal and $0$s elsewhere.

Matrix multiplication can have zero divisors. For example,

$$
A=\begin{bmatrix}1&-1\\1&-1\end{bmatrix},
\qquad
B=\begin{bmatrix}1&1\\1&1\end{bmatrix},
$$

yet

$$
AB=O.
$$

Thus $AB=O$ does not imply $A=O$ or $B=O$.


## Transpose

The transpose of $A=[a_{ij}]$ is $A^T=[a_{ji}]$: rows become columns.

If

$$
M=\begin{bmatrix}
9&20&21\\
1&5&7
\end{bmatrix},
$$

then

$$
M^T=\begin{bmatrix}
9&1\\20&5\\21&7
\end{bmatrix}.
$$

Important identities include

$$
(A^T)^T=A,
\qquad
(A+B)^T=A^T+B^T,
$$

$$
(cA)^T=cA^T,
\qquad
(AB)^T=B^TA^T.
$$

Notice the reversed order in the product identity.

In [ ]:
M=sp.Matrix([[9,20,21],[1,5,7]])
show_matrix(M,'M')
show_matrix(M.T,'M^T')
show_matrix((A_assoc*B_assoc).T,'(AB)^T')
show_matrix(B_assoc.T*A_assoc.T,'B^TA^T')
print('Transpose product identity verified:', (A_assoc*B_assoc).T == B_assoc.T*A_assoc.T)

> **Closing note**
>
> Matrix algebra resembles ordinary algebra in many ways, but multiplication has a directional structure. Associativity and distributivity survive; commutativity generally does not. The transpose also reverses multiplication order.

## Section 2.2 terminology

| Term | Meaning |
|---|---|
| Zero matrix $O$ | Matrix whose entries are all zero |
| Additive inverse $-A$ | Matrix satisfying $A+(-A)=O$ |
| Identity matrix $I_n$ | Square matrix with ones on the main diagonal |
| Associative property | Grouping does not change a valid product or sum |
| Distributive property | Multiplication distributes over addition |
| Commutative property | Order can be reversed; generally false for matrix multiplication |
| Transpose $A^T$ | Matrix formed by interchanging rows and columns |
| Symmetric matrix | Square matrix satisfying $A^T=A$ |

## Section 2.2 worksheet

1. Solve $4X-A=B$ for $X$, where

$$
A=\begin{bmatrix}4&8\\12&16\end{bmatrix},
\qquad
B=\begin{bmatrix}8&0\\4&20\end{bmatrix}.
$$

2. Verify $c(A+B)=cA+cB$ for

$$
A=\begin{bmatrix}1&2\\3&4\end{bmatrix},
\quad
B=\begin{bmatrix}5&-1\\2&0\end{bmatrix},
\quad c=3.
$$

3. Compute $AB$ and $BA$ for

$$
A=\begin{bmatrix}1&2\\0&1\end{bmatrix},
\qquad
B=\begin{bmatrix}2&0\\1&1\end{bmatrix}.
$$

4. Verify associativity for the matrices

$$
A=\begin{bmatrix}1&0\\2&1\end{bmatrix},
B=\begin{bmatrix}1&2\\0&1\end{bmatrix},
C=\begin{bmatrix}2&1\\1&0\end{bmatrix}.
$$

5. Find $M^T$ and $(M^T)^T$ for

$$
M=\begin{bmatrix}1&9&20\\21&0&5\end{bmatrix}.
$$

6. Verify $(AB)^T=B^TA^T$ for the matrices in Problem 3.

# Section 2.3 - The Inverse of a Matrix

## Learning objectives

Students should be able to:

- explain invertibility and singularity;
- find a $2\times2$ inverse using the formula;
- find an inverse using Gauss-Jordan elimination;
- recognize when an inverse does not exist;
- use inverse properties correctly;
- solve $A\mathbf{x}=\mathbf{b}$ with $A^{-1}$;
- reuse one inverse for several right-hand sides.

### Definition: inverse

> A square matrix $A$ is invertible when there exists a matrix $A^{-1}$ satisfying $AA^{-1}=A^{-1}A=I$. If no inverse exists, $A$ is singular. An inverse, when it exists, is unique.

## The $2\times2$ formula

For

$$
A=\begin{bmatrix}a&b\\c&d\end{bmatrix},
$$

$A$ is invertible exactly when $ad-bc\ne0$, and

$$
A^{-1}=\frac{1}{ad-bc}
\begin{bmatrix}d&-b\\-c&a\end{bmatrix}.
$$

### Original example

Let

$$
A=\begin{bmatrix}3&2\\4&3\end{bmatrix}.
$$

The determinant factor is

$$
ad-bc=3(3)-2(4)=1.
$$

Therefore,

$$
A^{-1}=\begin{bmatrix}3&-2\\-4&3\end{bmatrix}.
$$

Check:

$$
AA^{-1}
=\begin{bmatrix}3&2\\4&3\end{bmatrix}
\begin{bmatrix}3&-2\\-4&3\end{bmatrix}
=\begin{bmatrix}1&0\\0&1\end{bmatrix}.
$$

In [ ]:
A23_2=sp.Matrix([[3,2],[4,3]])
show_matrix(A23_2,'A')
print('det(A) =', A23_2.det())
show_matrix(A23_2.inv(),'A^{-1}')
show_matrix(A23_2*A23_2.inv(),'AA^{-1}')

## Gauss-Jordan method for a $3\times3$ inverse

Let

$$
A=\begin{bmatrix}
1&1&0\\
0&1&1\\
1&0&1
\end{bmatrix}.
$$

Adjoin the identity matrix:

$$
\left[\begin{array}{ccc|ccc}
1&1&0&1&0&0\\
0&1&1&0&1&0\\
1&0&1&0&0&1
\end{array}\right].
$$

### Step 1: eliminate the first entry of Row 3

Use $R_3\leftarrow R_3-R_1$:

$$
\left[\begin{array}{ccc|ccc}
1&1&0&1&0&0\\
0&1&1&0&1&0\\
0&-1&1&-1&0&1
\end{array}\right].
$$

### Step 2: eliminate the second entry of Row 3

Use $R_3\leftarrow R_3+R_2$:

$$
\left[\begin{array}{ccc|ccc}
1&1&0&1&0&0\\
0&1&1&0&1&0\\
0&0&2&-1&1&1
\end{array}\right].
$$

### Step 3: scale Row 3

Use $R_3\leftarrow\frac12R_3$:

$$
\left[\begin{array}{ccc|ccc}
1&1&0&1&0&0\\
0&1&1&0&1&0\\
0&0&1&-\frac12&\frac12&\frac12
\end{array}\right].
$$

### Step 4: eliminate the third entry of Row 2

Use $R_2\leftarrow R_2-R_3$:

$$
\left[\begin{array}{ccc|ccc}
1&1&0&1&0&0\\
0&1&0&\frac12&\frac12&-\frac12\\
0&0&1&-\frac12&\frac12&\frac12
\end{array}\right].
$$

### Step 5: eliminate the second entry of Row 1

Use $R_1\leftarrow R_1-R_2$:

$$
\left[\begin{array}{ccc|ccc}
1&0&0&\frac12&-\frac12&\frac12\\
0&1&0&\frac12&\frac12&-\frac12\\
0&0&1&-\frac12&\frac12&\frac12
\end{array}\right].
$$

Thus

$$
A^{-1}=\frac12
\begin{bmatrix}
1&-1&1\\
1&1&-1\\
-1&1&1
\end{bmatrix}.
$$

In [ ]:
A23=sp.Matrix([[1,1,0],[0,1,1],[1,0,1]])
M0=A23.row_join(sp.eye(3))
show_augmented(M0,3,r'[A\mid I]')
M1=M0.copy(); M1[2,:]=M1[2,:]-M1[0,:]
show_augmented(M1,3,r'R_3\leftarrow R_3-R_1')
M2=M1.copy(); M2[2,:]=M2[2,:]+M2[1,:]
show_augmented(M2,3,r'R_3\leftarrow R_3+R_2')
M3=M2.copy(); M3[2,:]=sp.Rational(1,2)*M3[2,:]
show_augmented(M3,3,r'R_3\leftarrow \frac12R_3')
M4=M3.copy(); M4[1,:]=M4[1,:]-M4[2,:]
show_augmented(M4,3,r'R_2\leftarrow R_2-R_3')
M5=M4.copy(); M5[0,:]=M5[0,:]-M5[1,:]
show_augmented(M5,3,r'R_1\leftarrow R_1-R_2')
show_matrix(A23.inv(),'A^{-1}')

## Singular matrices

For

$$
S=\begin{bmatrix}1&2\\2&4\end{bmatrix},
$$

$ad-bc=4-4=0$. The second row is twice the first, so the matrix cannot be reduced to $I_2$. Therefore, $S$ is singular.

A square matrix is **singular** precisely when it is not invertible. Equivalently,

$$
A\text{ singular}\iff\det(A)=0.
$$


In [ ]:
Ssing=sp.Matrix([[1,2],[2,4]])
print('det(S) =', Ssing.det())
print('rank(S) =', Ssing.rank())
print('RREF:', Ssing.rref()[0])

## Properties of inverses

For invertible matrices $A$ and $B$ and nonzero scalar $c$:

$$
(A^{-1})^{-1}=A,
\qquad
(cA)^{-1}=\frac1cA^{-1},
$$

$$
(A^T)^{-1}=(A^{-1})^T,
\qquad
(AB)^{-1}=B^{-1}A^{-1}.
$$

The product rule reverses order.

## Solving a system with an inverse

Use the same matrix

$$
A=\begin{bmatrix}1&1&0\\0&1&1\\1&0&1\end{bmatrix}
$$

and solve

$$
A\mathbf{x}=\begin{bmatrix}29\\41\\30\end{bmatrix}.
$$

Multiply by $A^{-1}$:

$$
\mathbf{x}=A^{-1}\mathbf{b}
=\frac12
\begin{bmatrix}1&-1&1\\1&1&-1\\-1&1&1\end{bmatrix}
\begin{bmatrix}29\\41\\30\end{bmatrix}.
$$

Compute each entry:

$$
\begin{aligned}
x_1&=\frac12(29-41+30)=9,\\
x_2&=\frac12(29+41-30)=20,\\
x_3&=\frac12(-29+41+30)=21.
\end{aligned}
$$

Therefore,

$$
\mathbf{x}=\begin{bmatrix}9\\20\\21\end{bmatrix}.
$$

In [ ]:
b23=sp.Matrix([29,41,30])
x23=A23.inv()*b23
show_matrix(b23,r'\mathbf{b}')
show_matrix(x23,r'A^{-1}\mathbf{b}')
show_matrix(A23*x23,r'A\mathbf{x}')

> **Multiple right-hand sides**
>
> If the same coefficient matrix is used with many right-hand-side vectors, computing the inverse once allows all solutions to be organized in one matrix equation.

> **Closing note**
>
> An inverse reverses the action of an invertible matrix. The Gauss-Jordan method explains both how to find the inverse and how singularity appears when the left block cannot become the identity.

## Section 2.3 terminology

| Term | Meaning |
|---|---|
| Invertible / nonsingular | Matrix with a multiplicative inverse |
| Singular | Matrix without an inverse |
| Inverse $A^{-1}$ | Matrix satisfying $AA^{-1}=A^{-1}A=I$ |
| Adjoining $I$ | Forming $[A\mid I]$ for Gauss-Jordan inversion |
| Determinant factor $ad-bc$ | Test used in the $2\times2$ inverse formula |
| Inverse of a product | $(AB)^{-1}=B^{-1}A^{-1}$ |
| Cancellation | Valid when the cancelled matrix is invertible |

## Section 2.3 worksheet

1. Find the inverse of

$$
A=\begin{bmatrix}2&1\\5&3\end{bmatrix}
$$

using the $2\times2$ formula and verify the result.

2. Determine whether

$$
B=\begin{bmatrix}3&6\\1&2\end{bmatrix}
$$

is invertible.

3. Find the inverse of

$$
C=\begin{bmatrix}1&1&0\\0&1&1\\1&1&1\end{bmatrix}
$$

by Gauss-Jordan elimination. Show every row operation.

4. Solve

$$
\begin{bmatrix}3&2\\4&3\end{bmatrix}
\begin{bmatrix}x\\y\end{bmatrix}
=\begin{bmatrix}9\\13\end{bmatrix}
$$

using an inverse.

5. Let

$$
P=\begin{bmatrix}1&1\\0&1\end{bmatrix},
\qquad
Q=\begin{bmatrix}1&0\\1&1\end{bmatrix}.
$$

Find $(PQ)^{-1}$ in two ways and verify that $(PQ)^{-1}=Q^{-1}P^{-1}$.

# Section 2.4 - Elementary Matrices and LU-Factorization

## Learning objectives

Students should be able to:

- construct elementary matrices;
- represent row operations by left multiplication;
- explain row equivalence using products of elementary matrices;
- factor an invertible matrix into elementary matrices;
- find an LU-factorization;
- solve $A\mathbf{x}=\mathbf{b}$ using forward and back substitution.

## Bridge: elementary matrices lead to LU

Every row operation can be represented by left multiplication with an elementary matrix. If elimination gives

$$
E_k\cdots E_2E_1A=U,
$$

then reversing the steps gives

$$
A=E_1^{-1}E_2^{-1}\cdots E_k^{-1}U.
$$

The product of the inverse elimination matrices is lower triangular; call it $L$. Therefore,

$$
A=LU.
$$

**LU factorization is Gaussian elimination with the elimination multipliers saved instead of discarded.**


### Definition: elementary matrix

> An elementary matrix is obtained by performing one elementary row operation on an identity matrix. Left multiplication by that elementary matrix performs the same row operation on any compatible matrix.

## The three types of elementary matrices

Starting with $I_3$:

1. Swap $R_1$ and $R_2$:

$$
E_{\text{swap}}=\begin{bmatrix}0&1&0\\1&0&0\\0&0&1\end{bmatrix}.
$$

2. Scale $R_2$ by $5$:

$$
E_{\text{scale}}=\begin{bmatrix}1&0&0\\0&5&0\\0&0&1\end{bmatrix}.
$$

3. Replace $R_3$ by $R_3-2R_1$:

$$
E_{\text{replace}}=\begin{bmatrix}1&0&0\\0&1&0\\-2&0&1\end{bmatrix}.
$$

In [ ]:
I3=sp.eye(3)
E_swap=sp.Matrix([[0,1,0],[1,0,0],[0,0,1]])
E_scale=sp.diag(1,5,1)
E_replace=sp.Matrix([[1,0,0],[0,1,0],[-2,0,1]])
show_matrix(E_swap,'E_{swap}')
show_matrix(E_scale,'E_{scale}')
show_matrix(E_replace,'E_{replace}')

### Left multiplication performs the row operation

Let

$$
A=\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix}.
$$

Then

$$
E_{\text{replace}}A
=\begin{bmatrix}1&2\\3&4\\3&2\end{bmatrix},
$$

which is exactly the result of $R_3\leftarrow R_3-2R_1$.

In [ ]:
A_elem=sp.Matrix([[1,2],[3,4],[5,6]])
show_matrix(A_elem,'A')
show_matrix(E_replace*A_elem,'E_{replace}A')

## Factoring a matrix into elementary matrices

Let

$$
A=\begin{bmatrix}1&2\\3&7\end{bmatrix}.
$$

Reduce $A$ to $I_2$:

$$
R_2\leftarrow R_2-3R_1
\quad\Longrightarrow\quad
\begin{bmatrix}1&2\\0&1\end{bmatrix}.
$$

Then

$$
R_1\leftarrow R_1-2R_2
\quad\Longrightarrow\quad
I_2.
$$

The corresponding elementary matrices are

$$
E_1=\begin{bmatrix}1&0\\-3&1\end{bmatrix},
\qquad
E_2=\begin{bmatrix}1&-2\\0&1\end{bmatrix}.
$$

Since $E_2E_1A=I$, we have

$$
A=E_1^{-1}E_2^{-1}
=\begin{bmatrix}1&0\\3&1\end{bmatrix}
\begin{bmatrix}1&2\\0&1\end{bmatrix}.
$$

In [ ]:
A_fact=sp.Matrix([[1,2],[3,7]])
E1=sp.Matrix([[1,0],[-3,1]])
E2=sp.Matrix([[1,-2],[0,1]])
show_matrix(E2*E1*A_fact,'E_2E_1A')
show_matrix(E1.inv()*E2.inv(),'E_1^{-1}E_2^{-1}')

## LU-factorization

An LU-factorization writes

$$
A=LU,
$$

where $L$ is lower triangular and $U$ is upper triangular.

Consider

$$
A=\begin{bmatrix}
2&1&-1\\
4&5&0\\
-2&8&11
\end{bmatrix}.
$$

### Elimination step 1

Use $R_2\leftarrow R_2-2R_1$. The multiplier is $m_{21}=2$:

$$
\begin{bmatrix}
2&1&-1\\
0&3&2\\
-2&8&11
\end{bmatrix}.
$$

### Elimination step 2

Use $R_3\leftarrow R_3+R_1=R_3-(-1)R_1$. The multiplier is $m_{31}=-1$:

$$
\begin{bmatrix}
2&1&-1\\
0&3&2\\
0&9&10
\end{bmatrix}.
$$

### Elimination step 3

Use $R_3\leftarrow R_3-3R_2$. The multiplier is $m_{32}=3$:

$$
U=\begin{bmatrix}
2&1&-1\\
0&3&2\\
0&0&4
\end{bmatrix}.
$$

Store the multipliers below the diagonal:

$$
L=\begin{bmatrix}
1&0&0\\
2&1&0\\
-1&3&1
\end{bmatrix}.
$$

Therefore, $A=LU$.

In [ ]:
A_lu=sp.Matrix([[2,1,-1],[4,5,0],[-2,8,11]])
L=sp.Matrix([[1,0,0],[2,1,0],[-1,3,1]])
U=sp.Matrix([[2,1,-1],[0,3,2],[0,0,4]])
show_matrix(A_lu,'A')
show_matrix(L,'L')
show_matrix(U,'U')
show_matrix(L*U,'LU')
print('A = LU?', A_lu == L*U)

### The two triangular solves

Once $A=LU$,

$$
A\mathbf x=\mathbf b
\quad\Longrightarrow\quad
LU\mathbf x=\mathbf b.
$$

Introduce $\mathbf y=U\mathbf x$. First solve

$$
L\mathbf y=\mathbf b
$$

by forward substitution, then solve

$$
U\mathbf x=\mathbf y
$$

by back substitution.


## Solving a system with LU

Solve

$$
A\mathbf{x}=\mathbf{b},
\qquad
\mathbf{b}=\begin{bmatrix}-9\\49\\290\end{bmatrix}.
$$

Because $A=LU$, first solve

$$
L\mathbf{y}=\mathbf{b}.
$$

This gives

$$
\begin{cases}
y_1=-9,\\
2y_1+y_2=49,\\
-y_1+3y_2+y_3=290.
\end{cases}
$$

Forward substitution:

$$
y_1=-9,
$$

$$
y_2=49-2(-9)=67,
$$

$$
y_3=290+y_1-3y_2=290-9-201=80.
$$

Thus

$$
\mathbf{y}=\begin{bmatrix}-9\\67\\80\end{bmatrix}.
$$

Now solve

$$
U\mathbf{x}=\mathbf{y}.
$$

The equations are

$$
\begin{cases}
2x_1+x_2-x_3=-9,\\
3x_2+2x_3=67,\\
4x_3=80.
\end{cases}
$$

Back substitution:

$$
x_3=20,
$$

$$
3x_2+40=67 \quad\Longrightarrow\quad x_2=9,
$$

$$
2x_1+9-20=-9 \quad\Longrightarrow\quad x_1=1.
$$

Therefore,

$$
\mathbf{x}=\begin{bmatrix}1\\9\\20\end{bmatrix}.
$$

In [ ]:
b_lu=sp.Matrix([-9,49,290])
y=L.inv()*b_lu
x=U.inv()*y
show_matrix(y,r'\mathbf{y}')
show_matrix(x,r'\mathbf{x}')
show_matrix(A_lu*x,r'A\mathbf{x}')

> **Why LU is useful**
>
> When many systems share the same coefficient matrix, factor $A=LU$ once. Each new right-hand side then requires only one forward substitution and one back substitution.

The main savings occurs when the same matrix $A$ is reused with many right-hand sides: elimination is done once, and each new system needs only two triangular solves.


> **Closing note**
>
> Elementary matrices convert row operations into matrix multiplication. LU-factorization records the elimination process in two triangular matrices, making repeated system solving efficient and conceptually transparent.

## Section 2.4 terminology

| Term | Meaning |
|---|---|
| Elementary matrix | Identity matrix after one elementary row operation |
| Row equivalent | Related by a finite sequence of elementary row operations |
| Lower triangular matrix | Entries above the main diagonal are zero |
| Upper triangular matrix | Entries below the main diagonal are zero |
| LU-factorization | Factorization $A=LU$ |
| Elimination multiplier | Number used to eliminate an entry below a pivot |
| Forward substitution | Solve a lower triangular system from top to bottom |
| Back substitution | Solve an upper triangular system from bottom to top |

## Section 2.4 worksheet

1. Construct the $3\times3$ elementary matrix that performs each operation:

- $R_1\leftrightarrow R_3$
- $R_2\leftarrow -2R_2$
- $R_3\leftarrow R_3+4R_1$

2. Let

$$
A=\begin{bmatrix}1&2\\3&5\\4&7\end{bmatrix}.
$$

Use an elementary matrix to perform $R_3\leftarrow R_3-4R_1$ and verify by multiplication.

3. Factor

$$
B=\begin{bmatrix}1&3\\2&7\end{bmatrix}
$$

as a product of elementary matrices.

4. Find an LU-factorization of

$$
C=\begin{bmatrix}2&1&0\\4&3&1\\-2&2&4\end{bmatrix}.
$$

5. Use your factorization from Problem 4 to solve

$$
C\mathbf{x}=\begin{bmatrix}4\\11\\14\end{bmatrix}.
$$

# Section 2.5 - Markov Chains

## Learning objectives

Students should be able to:

- recognize a column-stochastic matrix;
- compute state vectors using $\mathbf{x}_{k+1}=P\mathbf{x}_k$;
- compute an $n$th state using $P^n\mathbf{x}_0$;
- find a steady state from $P\mathbf{q}=\mathbf{q}$;
- identify regular and absorbing chains;
- interpret long-run behavior in applications.

> **Convention used in this course**
>
> State vectors are columns. The entry p(i,j) is the probability of moving from state j to state i. Therefore, each column of a transition matrix sums to 1, and the next state is found by multiplying P by the current state.

## Stochastic matrices

A column-stochastic matrix has nonnegative entries and each column sums to $1$.

### Original two-state R&B/pop model

Suppose listeners are classified as primarily choosing R&B or pop during a weekly listening window. Use

$$
P=\begin{bmatrix}
0.80&0.30\\
0.20&0.70
\end{bmatrix}.
$$

Column 1 describes what happens to current R&B listeners; column 2 describes what happens to current pop listeners.

Start from

$$
\mathbf{x}_0=\begin{bmatrix}0.21\\0.79\end{bmatrix}.
$$

Then

$$
\mathbf{x}_1=P\mathbf{x}_0
=\begin{bmatrix}
0.80(0.21)+0.30(0.79)\\
0.20(0.21)+0.70(0.79)
\end{bmatrix}
=\begin{bmatrix}0.405\\0.595\end{bmatrix}.
$$

This course uses

$$
p_{ij}=P(\text{next state }i\mid\text{current state }j),
$$

so **columns are starting states** and **rows are next states**. The first column therefore says that a current R&B listener has an $80\%$ chance to remain in R&B and a $20\%$ chance to switch to pop at the next step.

Also, $\mathbf x_1=P\mathbf x_0$ is the **next time step** after the initial distribution.


In [ ]:
P_music=sp.Matrix([[sp.Rational(4,5),sp.Rational(3,10)],
                   [sp.Rational(1,5),sp.Rational(7,10)]])
x0_music=sp.Matrix([sp.Rational(21,100),sp.Rational(79,100)])
show_matrix(P_music,'P')
show_matrix(x0_music,r'\mathbf{x}_0')
for n in [1,2,3,5,10]:
    show_matrix(sp.N(P_music**n*x0_music,6),rf'\mathbf{{x}}_{n}')

In [ ]:
# Convergence plot for the two-state chain.
P_np=np.array(P_music.tolist(),dtype=float)
x=np.array(x0_music,dtype=float).reshape(-1)
history=[x.copy()]
for _ in range(20):
    x=P_np@x
    history.append(x.copy())
history=np.array(history)
plt.figure(figsize=(7,4))
plt.plot(history[:,0],marker='o',label='R&B proportion')
plt.plot(history[:,1],marker='s',label='Pop proportion')
plt.xlabel('Week')
plt.ylabel('Proportion')
plt.ylim(0,1)
plt.grid(alpha=.3)
plt.legend()
plt.show()

## Steady state

A steady state $\mathbf{q}$ satisfies

$$
P\mathbf{q}=\mathbf{q},
\qquad
q_1+q_2=1.
$$

For the music model,

$$
\begin{cases}
0.8q_1+0.3q_2=q_1,\\
q_1+q_2=1.
\end{cases}
$$

The first equation gives

$$
0.2q_1=0.3q_2
\quad\Longrightarrow\quad
2q_1=3q_2.
$$

Together with $q_1+q_2=1$,

$$
\mathbf{q}=\begin{bmatrix}0.6\\0.4\end{bmatrix}.
$$

In [ ]:
q1,q2=sp.symbols('q1 q2')
sol=sp.solve([
    sp.Eq(sp.Rational(4,5)*q1+sp.Rational(3,10)*q2,q1),
    sp.Eq(q1+q2,1)
],[q1,q2])
print(sol)
q=sp.Matrix([sol[q1],sol[q2]])
show_matrix(P_music*q,r'P\mathbf{q}')
show_matrix(q,r'\mathbf{q}')

## Three-state travel-mode chain

Let the states be driving, public transit, and walking. Consider

$$
P=\begin{bmatrix}
0.7&0.2&0.1\\
0.2&0.6&0.3\\
0.1&0.2&0.6
\end{bmatrix},
\qquad
\mathbf{x}_0=\begin{bmatrix}0.5\\0.3\\0.2\end{bmatrix}.
$$

The steady state is

$$
\mathbf{q}=\begin{bmatrix}10/29\\11/29\\8/29\end{bmatrix}
\approx
\begin{bmatrix}0.3448\\0.3793\\0.2759\end{bmatrix}.
$$

In [ ]:
P_travel=sp.Matrix([[sp.Rational(7,10),sp.Rational(1,5),sp.Rational(1,10)],
                    [sp.Rational(1,5),sp.Rational(3,5),sp.Rational(3,10)],
                    [sp.Rational(1,10),sp.Rational(1,5),sp.Rational(3,5)]])
x0_travel=sp.Matrix([sp.Rational(1,2),sp.Rational(3,10),sp.Rational(1,5)])
for n in [1,2,5,10,25]:
    show_matrix(sp.N(P_travel**n*x0_travel,6),rf'\mathbf{{x}}_{n}')
v=(P_travel-sp.eye(3)).nullspace()[0]
v=sp.simplify(v/sum(v))
show_matrix(v,r'\mathbf{q}')

## Absorbing Markov chain: hiking outcomes

Use three states:

1. still on the trail;
2. reached the summit;
3. returned to the trailhead.

Suppose

$$
P=\begin{bmatrix}
0.5&0&0\\
0.3&1&0\\
0.2&0&1
\end{bmatrix}.
$$

The summit and trailhead-return states are absorbing. Starting on the trail,

$$
\mathbf{x}_0=\begin{bmatrix}1\\0\\0\end{bmatrix}.
$$

After one transition,

$$
\mathbf{x}_1=\begin{bmatrix}0.5\\0.3\\0.2\end{bmatrix}.
$$

After two,

$$
\mathbf{x}_2=\begin{bmatrix}0.25\\0.45\\0.30\end{bmatrix}.
$$

The probability of still being on the trail is $(1/2)^n$, so it approaches zero. The limiting absorption probabilities are

$$
\lim_{n\to\infty}\mathbf{x}_n
=\begin{bmatrix}0\\0.6\\0.4\end{bmatrix}.
$$

Here “absorbing” means a **terminal recorded outcome**, not that a hiker is physically trapped forever. Once the experiment records “reached the summit” or “returned to the trailhead,” that trial is finished.


In [ ]:
P_hike=sp.Matrix([[sp.Rational(1,2),0,0],
                  [sp.Rational(3,10),1,0],
                  [sp.Rational(1,5),0,1]])
x0_hike=sp.Matrix([1,0,0])
for n in [1,2,3,5,10]:
    show_matrix(sp.N(P_hike**n*x0_hike,6),rf'\mathbf{{x}}_{n}')

In [ ]:
# Directed graph for the hiking chain.
fig, ax = plt.subplots(figsize=(7,4))
ax.axis('off')
pos={'Trail':(0.15,0.5),'Summit':(0.75,0.75),'Return':(0.75,0.25)}
for name,(x,y) in pos.items():
    circ=plt.Circle((x,y),0.09,fill=False,linewidth=2)
    ax.add_patch(circ); ax.text(x,y,name,ha='center',va='center')

def arrow(a,b,label,rad=0.0):
    ax.annotate('',xy=pos[b],xytext=pos[a],arrowprops=dict(arrowstyle='->',connectionstyle=f'arc3,rad={rad}',lw=1.8))
    mx=(pos[a][0]+pos[b][0])/2; my=(pos[a][1]+pos[b][1])/2
    ax.text(mx,my+0.04,label,ha='center')
arrow('Trail','Summit','0.3')
arrow('Trail','Return','0.2')
# self loops as text labels for clarity
ax.text(0.07,0.65,'stay 0.5')
ax.text(0.78,0.90,'absorbing 1')
ax.text(0.78,0.08,'absorbing 1')
ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.show()

> **Closing note**
>
> Markov chains turn repeated probabilistic transitions into matrix powers. A steady state is a balance vector that is unchanged by the transition matrix. Absorbing chains model eventual outcomes rather than long-run mixing.

## Section 2.5 terminology

| Term | Meaning |
|---|---|
| State | Category occupied by a member of the population |
| Transition probability | Probability of moving from one state to another |
| Stochastic matrix | Nonnegative matrix whose columns sum to $1$ |
| State vector | Column of state proportions or probabilities |
| Markov chain | Repeated process governed by a fixed transition matrix |
| $n$th state | $\mathbf{x}_n=P^n\mathbf{x}_0$ |
| Steady state | Probability vector satisfying $P\mathbf{q}=\mathbf{q}$ |
| Regular matrix | Stochastic matrix with some positive power |
| Absorbing state | State with probability $1$ of remaining there |
| Absorbing chain | Chain that can eventually reach an absorbing state |

## Section 2.5 worksheet

1. Determine whether each matrix is column-stochastic:

$$
A=\begin{bmatrix}0.7&0.2\\0.3&0.8\end{bmatrix},
\qquad
B=\begin{bmatrix}0.6&0.5\\0.5&0.5\end{bmatrix}.
$$

2. For

$$
P=\begin{bmatrix}0.75&0.25\\0.25&0.75\end{bmatrix},
\qquad
\mathbf{x}_0=\begin{bmatrix}0.8\\0.2\end{bmatrix},
$$

find $\mathbf{x}_1$, $\mathbf{x}_2$, and $\mathbf{x}_5$.

3. Find the steady state of the matrix in Problem 2.

4. For the three-state matrix

$$
P=\begin{bmatrix}0.6&0.2&0.1\\0.3&0.7&0.2\\0.1&0.1&0.7\end{bmatrix},
$$

verify that it is stochastic and find the next state from

$$
\mathbf{x}_0=\begin{bmatrix}0.4\\0.4\\0.2\end{bmatrix}.
$$

5. Consider the absorbing chain

$$
P=\begin{bmatrix}0.4&0&0\\0.35&1&0\\0.25&0&1\end{bmatrix}.
$$

Identify the absorbing states and find the limiting absorption probabilities when the process starts in State 1.

# Section 2.6 - More Applications of Matrix Operations

## Learning objectives

Students should be able to:

- encode and decode a message using an invertible matrix;
- formulate and solve an open Leontief input-output model;
- derive the normal equations for least squares regression;
- compute and interpret a least squares line and residual error.

## Matrix cryptography

Use the mapping

$$
\text{blank}=0,\quad A=1,\quad B=2,\quad\ldots,\quad Z=26.
$$

Encode row vectors on the right using

$$
E=\begin{bmatrix}1&2\\1&3\end{bmatrix}.
$$

Because

$$
\det(E)=1(3)-2(1)=1,
$$

$E$ is invertible and

$$
E^{-1}=\begin{bmatrix}3&-2\\-1&1\end{bmatrix}.
$$

### Original message: GO SPURS GO

The numerical sequence is

$$
7,15,0,19,16,21,18,19,0,7,15,0.
$$

Partition into pairs:

$$
[7\ 15],\ [0\ 19],\ [16\ 21],\ [18\ 19],\ [0\ 7],\ [15\ 0].
$$

Encode each pair:

$$
[7\ 15]E=[22\ 59],
$$

$$
[0\ 19]E=[19\ 57],
$$

$$
[16\ 21]E=[37\ 95],
$$

$$
[18\ 19]E=[37\ 93],
$$

$$
[0\ 7]E=[7\ 21],
$$

$$
[15\ 0]E=[15\ 30].
$$

The cryptogram is

$$
22,59,19,57,37,95,37,93,7,21,15,30.
$$

To decode, multiply each coded row by $E^{-1}$ on the right.

In [ ]:
letter_to_num={' ':0, **{chr(64+i):i for i in range(1,27)}}
num_to_letter={v:k for k,v in letter_to_num.items()}
E=sp.Matrix([[1,2],[1,3]])
Einv=E.inv()

def encode_message(message, matrix=E):
    nums=[letter_to_num[ch] for ch in message.upper()]
    if len(nums)%2: nums.append(0)
    coded=[]
    for i in range(0,len(nums),2):
        coded.extend(list(sp.Matrix([[nums[i],nums[i+1]]])*matrix))
    return nums, [int(v) for v in coded]

def decode_message(coded, inverse=Einv):
    nums=[]
    for i in range(0,len(coded),2):
        nums.extend(list(sp.Matrix([[coded[i],coded[i+1]]])*inverse))
    return ''.join(num_to_letter[int(v)] for v in nums)

nums,coded=encode_message('GO SPURS GO')
print('Numeric message:',nums)
print('Coded:',coded)
print('Decoded:',repr(decode_message(coded)))
show_matrix(E,'E')
show_matrix(Einv,'E^{-1}')

## Leontief input-output model

Consider three fictional industries:

1. travel services;
2. music and entertainment;
3. sporting goods.

Let

$$
D=\begin{bmatrix}
0.10&0.05&0.10\\
0.20&0.10&0.05\\
0.05&0.10&0.10
\end{bmatrix}
$$

be the internal consumption matrix. If $\mathbf{x}$ is total output and $\mathbf{e}$ is external demand, then

$$
\mathbf{x}=D\mathbf{x}+\mathbf{e}.
$$

Therefore,

$$
(I-D)\mathbf{x}=\mathbf{e}.
$$

Suppose

$$
\mathbf{e}=\begin{bmatrix}95\\62\\56\end{bmatrix}.
$$

Then

$$
I-D=
\begin{bmatrix}
0.90&-0.05&-0.10\\
-0.20&0.90&-0.05\\
-0.05&-0.10&0.90
\end{bmatrix}.
$$

Multiplying by $20$ produces an equivalent integer system:

$$
\begin{bmatrix}
18&-1&-2\\
-4&18&-1\\
-1&-2&18
\end{bmatrix}
\mathbf{x}
=\begin{bmatrix}1900\\1240\\1120\end{bmatrix}.
$$

Solving gives

$$
\mathbf{x}=\begin{bmatrix}120\\100\\80\end{bmatrix}.
$$

Verification:

$$
D\mathbf{x}+\mathbf{e}
=\begin{bmatrix}25\\38\\24\end{bmatrix}
+\begin{bmatrix}95\\62\\56\end{bmatrix}
=\begin{bmatrix}120\\100\\80\end{bmatrix}.
$$

The scaled system can also be shown as

$$
\left[
\begin{array}{ccc|c}
18&-1&-2&1900\\
-4&18&-1&1240\\
-1&-2&18&1120
\end{array}
\right].
$$

For the first industry,

$$
0.10(120)+0.05(100)+0.10(80)=12+5+8=25
$$

units are used internally, and

$$
25+95=120
$$

verifies the first total-output component.


In [ ]:
D_io=sp.Matrix([[sp.Rational(1,10),sp.Rational(1,20),sp.Rational(1,10)],
                [sp.Rational(1,5),sp.Rational(1,10),sp.Rational(1,20)],
                [sp.Rational(1,20),sp.Rational(1,10),sp.Rational(1,10)]])
e_io=sp.Matrix([95,62,56])
M_io=sp.eye(3)-D_io
x_io=M_io.inv()*e_io
show_matrix(D_io,'D')
show_matrix(M_io,'I-D')
show_matrix(x_io,r'\mathbf{x}')
show_matrix(D_io*x_io+e_io,r'D\mathbf{x}+\mathbf{e}')

## Least squares regression

Suppose an original travel-related dataset is

$$
(1,8),\ (2,9),\ (3,17),\ (4,17),\ (5,24).
$$

Fit

$$
y=a_0+a_1x.
$$

Write

$$
X=\begin{bmatrix}
1&1\\
1&2\\
1&3\\
1&4\\
1&5
\end{bmatrix},
\qquad
\mathbf{y}=\begin{bmatrix}8\\9\\17\\17\\24\end{bmatrix}.
$$

The normal equations are

$$
X^TX\mathbf{a}=X^T\mathbf{y}.
$$

Compute

$$
X^TX=\begin{bmatrix}5&15\\15&55\end{bmatrix},
\qquad
X^T\mathbf{y}=\begin{bmatrix}75\\265\end{bmatrix}.
$$

Thus

$$
\begin{cases}
5a_0+15a_1=75,\\
15a_0+55a_1=265.
\end{cases}
$$

Multiply the first equation by $3$:

$$
15a_0+45a_1=225.
$$

Subtract from the second equation:

$$
10a_1=40,
\qquad
a_1=4.
$$

Then

$$
5a_0+15(4)=75
\quad\Longrightarrow\quad
a_0=3.
$$

The least squares line is

$$
\widehat{y}=3+4x.
$$

Predicted values are $7,11,15,19,23$, so the residual vector is

$$
\mathbf{r}=\begin{bmatrix}1\\-2\\2\\-2\\1\end{bmatrix}.
$$

The sum of squared errors is

$$
\mathrm{SSE}=1^2+(-2)^2+2^2+(-2)^2+1^2=14.
$$

In [ ]:
xs=np.array([1,2,3,4,5],dtype=float)
ys=np.array([8,9,17,17,24],dtype=float)
X=sp.Matrix([[1,int(x)] for x in xs])
yvec=sp.Matrix([int(y) for y in ys])
beta=(X.T*X).inv()*X.T*yvec
res=yvec-X*beta
sse=(res.T*res)[0]
show_matrix(X.T*X,'X^TX')
show_matrix(X.T*yvec,r'X^T\mathbf{y}')
show_matrix(beta,r'\mathbf{a}')
show_matrix(res,r'\mathbf{r}')
print('SSE =',sse)
plt.figure(figsize=(7,4.5))
plt.scatter(xs,ys,s=70,label='Observed data')
xx=np.linspace(.5,5.5,200)
plt.plot(xx,3+4*xx,label=r'$\hat y=3+4x$')
for x,y in zip(xs,ys):
    plt.plot([x,x],[y,3+4*x],linestyle='--',linewidth=1)
plt.xlabel('Input x')
plt.ylabel('Response y')
plt.grid(alpha=.3)
plt.legend()
plt.show()

### Optional self-exploration

Change the data passed to `fit_and_plot` and compare the fitted line and SSE.


In [ ]:
# Optional exploration: change the y-values and rerun.
def fit_and_plot(x_values, y_values):
    x_values=np.asarray(x_values,dtype=float)
    y_values=np.asarray(y_values,dtype=float)
    design=np.column_stack([np.ones(len(x_values)),x_values])
    coef=np.linalg.lstsq(design,y_values,rcond=None)[0]
    predictions=design@coef
    sse=float(np.sum((y_values-predictions)**2))
    print(f'intercept = {coef[0]:.4f}, slope = {coef[1]:.4f}, SSE = {sse:.4f}')
    grid=np.linspace(x_values.min()-.5,x_values.max()+.5,200)
    plt.figure(figsize=(7,4))
    plt.scatter(x_values,y_values,s=70)
    plt.plot(grid,coef[0]+coef[1]*grid)
    plt.grid(alpha=.3)
    plt.show()

fit_and_plot([1,2,3,4,5],[8,9,17,17,24])

> **Closing note**
>
> Cryptography uses invertible matrix transformations, Leontief models use $(I-D)^{-1}$ to account for internal demand, and least squares uses matrix projection ideas to fit inconsistent data. These applications show how the same matrix operations support very different fields.

## Section 2.6 terminology

| Term | Meaning |
|---|---|
| Cryptogram | Encoded numerical message |
| Encoding matrix | Invertible matrix used to transform message blocks |
| Decoding matrix | Inverse of the encoding matrix |
| Input-output matrix $D$ | Internal consumption coefficients in a Leontief model |
| External demand $\mathbf{e}$ | Demand from outside the modeled industries |
| Total output $\mathbf{x}$ | Production required to meet internal and external demand |
| Least squares line | Line minimizing the sum of squared residuals |
| Design matrix $X$ | Matrix containing regression predictor structure |
| Normal equations | $X^TX\mathbf{a}=X^T\mathbf{y}$ |
| Residual | Observed value minus predicted value |
| SSE | Sum of squared errors |

## Section 2.6 worksheet

1. Use the mapping blank $=0$, $A=1$, ..., $Z=26$ and

$$
E=\begin{bmatrix}1&1\\1&2\end{bmatrix}
$$

to encode the message **RNB MUSIC**. Pad with a blank if necessary.

2. Find $E^{-1}$ and decode the first two coded blocks from Problem 1.

3. A fictional two-industry economy has

$$
D=\begin{bmatrix}0.20&0.10\\0.25&0.20\end{bmatrix},
\qquad
\mathbf{e}=\begin{bmatrix}70\\55\end{bmatrix}.
$$

Solve $(I-D)\mathbf{x}=\mathbf{e}$.

4. Find the least squares line for

$$
(1,4),\ (2,7),\ (3,8),\ (4,13).
$$

Show $X$, $X^TX$, $X^T\mathbf{y}$, the normal equations, and the SSE.

# Chapter 2 Summary

The chapter develops a connected sequence:

$$
\text{matrix operations}
\longrightarrow
\text{matrix algebra}
\longrightarrow
\text{inverses and elementary matrices}
\longrightarrow
\text{LU-factorization}
\longrightarrow
\text{dynamic and applied models}.
$$

Core computational habits:

1. check matrix dimensions before every operation;
2. show row-column products explicitly when multiplying;
3. verify inverses by multiplication;
4. record every elementary row operation;
5. keep the order of matrix products intact;
6. interpret matrix results in the original application context;
7. use Python to verify, not replace, mathematical reasoning.